In [0]:
# =====================================================
# 1. DEFINE ADF WIDGET PARAMETERS
# =====================================================

dbutils.widgets.text("landing_timestamp", "", "")
dbutils.widgets.text("table_name", "", "")
dbutils.widgets.text("merge_keys", "", "")
dbutils.widgets.text("load_type", "", "")

# Get parameters from ADF
landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

In [0]:
print("[step 1]Reading source bronze table ")
source_query = f"SELECT * FROM adb_carsync.bronze.{table_name} WHERE landing_timestamp = '{landing_timestamp}'"
print(f"[source_query]\n{source_query}")

bronze_df=spark.read.table(f"adb_carsync.bronze.{table_name}").filter(
    f"landing_timestamp = '{landing_timestamp}'"
)

print(
    f"[STEP 1 completed]source dataframe perpared for table '{table_name}' with landing_timestamp = '{landing_timestamp}' "
)
print(f"[source columns ]{bronze_df.columns}")

In [0]:

# =====================================================
# 3. PREPARE LOAD METADATA
# =====================================================

import re
from pyspark.sql import functions as F

print("\n========== PREPARING LOAD METADATA ==========")

# Define Silver target table
target_table = f"adb_carsync.silver.{table_name}"

# Normalize load type
normalized_load_type = (load_type or "FULL").strip().upper()

# Prepare merge keys
merge_key_list = [
    key.strip()
    for key in merge_keys.split(",")
    if key.strip()
]

print("Target Table:", target_table)
print("Load Type:", normalized_load_type)
print("Merge Keys:", merge_key_list)

# Validate merge keys
if normalized_load_type == "MERGE":

    print("[MERGE VALIDATION] Validating merge keys")

    if not merge_key_list:
        raise ValueError(
            "Merge keys must be provided when load_type is 'MERGE'"
        )

    missing_merge_keys = [
        key for key in merge_key_list
        if key not in bronze_df.columns
    ]

    if missing_merge_keys:
        raise ValueError(
            f"Merge keys not found in source data: {missing_merge_keys}"
        )

    print("[MERGE VALIDATION COMPLETE] All merge keys are present in source data")

# Add Silver audit columns
updated_df = (
    bronze_df
    .withColumn("insert_timestamp", F.current_timestamp())
    .withColumn("updated_timestamp", F.current_timestamp())
)


print("Audit columns added successfully!")

# Calculate record count
record_count = updated_df.count()

print("Processed Record Count:", record_count)

print("\n========== LOAD METADATA PREPARED SUCCESSFULLY ==========")


In [0]:

# =====================================================
# 4. WRITE DATA TO SILVER BASED ON LOAD TYPE
# =====================================================

print("\n========== STARTING SILVER LOAD ==========")

print("Target Table:", target_table)
print("Load Type:", normalized_load_type)
print("Processed Record Count:", record_count)


if normalized_load_type == "FULL":

    print("\n[STEP 1] FULL LOAD STARTED")
    print("Write Mode: overwrite")

    updated_df.write \
        .mode("overwrite") \
        .saveAsTable(target_table)

    print("[STEP 1 COMPLETED] FULL load completed successfully")


elif normalized_load_type == "APPEND":

    print("\n[STEP 2] APPEND LOAD STARTED")
    print("Write Mode: append")

    updated_df.write \
        .mode("append") \
        .saveAsTable(target_table)

    print("[STEP 2 COMPLETED] APPEND load completed successfully")


elif normalized_load_type == "MERGE":

    print("\n[STEP 3] MERGE LOAD STARTED")
    print("Merge Keys:", merge_key_list)

    if not spark.catalog.tableExists(target_table):

        print("Silver table does not exist.")
        print("Creating new Silver table")

        updated_df.write \
            .mode("overwrite") \
            .saveAsTable(target_table)

        print("[STEP 3 COMPLETED] Initial MERGE load completed successfully")

    else:

        print("Silver table already exists.")
        print("Preparing MERGE operation")

        source_view = "silver_merge_source"

        updated_df.createOrReplaceTempView(source_view)

        print("Source temporary view created:", source_view)

        merge_condition = " AND ".join(
            [
                f"target.{key} = source.{key}"
                for key in merge_key_list
            ]
        )

        print("Merge Condition:", merge_condition)

        update_assignments = ", ".join(
            [
                f"target.{column} = source.{column}"
                for column in updated_df.columns
                if column != "insert_timestamp"
            ]
        )

        print("Update Assignments:", update_assignments)

        insert_columns = ", ".join(
            [f"`{column}`" for column in updated_df.columns]
        )

        insert_values = ", ".join(
            [f"source.`{column}`" for column in updated_df.columns]
        )

        merge_sql = f"""
            MERGE INTO {target_table} AS target
            USING {source_view} AS source
            ON {merge_condition}

            WHEN MATCHED THEN
                UPDATE SET {update_assignments}

            WHEN NOT MATCHED THEN
                INSERT ({insert_columns})
                VALUES ({insert_values})
        """

        print("\n[MERGE SQL]")
        print(merge_sql)

        print("Executing MERGE operation")

        spark.sql(merge_sql)

        print("[STEP 3 COMPLETED] MERGE load completed successfully")


else:

    raise ValueError(
        "load_type must be one of: FULL, APPEND, MERGE"
    )


print("\n========== SILVER LOAD COMPLETED SUCCESSFULLY ==========")
print("Target Table:", target_table)
print("Load Type:", normalized_load_type)
print("Processed Record Count:", record_count)


In [0]:
# =====================================================
# 5. RETURN PROCESSED RECORD COUNT
# =====================================================

record_count = updated_df.count()

print("Processed Record Count:", record_count)

dbutils.notebook.exit(str(record_count))